In [4]:
import numpy as np
import pandas as pd

In [5]:
df = pd.read_csv("10 DateFruit_Dataset.csv")

In [6]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [7]:
df.shape

(898, 35)

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]
X, y

In [9]:
df["Class"].unique()
#we got 7 unique classes so output layer will have 7 neurons (this is multiclass classification)

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [10]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ANN

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [14]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [15]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_train_tensor, y_train_tensor)

In [16]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [22]:
class ANN(nn.Module):
  def __init__(self):
    super(ANN, self).__init__()

    self.model = nn.Sequential(
        #layer1
        nn.Linear(X.shape[1], 64),
        nn.ReLU(),
        #layer2
        nn.Linear(64, 64),
        nn.ReLU(),
        #o/p layer
        nn.Linear(64, 7)
    )

  def forward(self, x):
    return self.model(x)

In [23]:
model = ANN()

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [38]:
# Training
best_loss = float("inf")

epochs = 100
for epoch in range(epochs):
  model.train()
  running_loss = 0.0
  for xb, yb in train_loader:
    optimizer.zero_grad()

    outputs = model(xb)
    loss = criteria(outputs, yb)
    loss.backward()
    optimizer.step() # wt update

    running_loss += loss.item() # converts from tensor to float

  train_loss = running_loss / len(train_loader)

  print(f"epoch= {epoch+1}/{epochs}, loss = {train_loss}")

# --- Save best model ---
  if train_loss < best_loss:
      best_loss = train_loss
      torch.save(model.state_dict(), "best_clsfictn_model.pth")
      print(f"Saved best model at epoch {epoch+1} with val loss {train_loss:.4f}")

epoch= 1/100, loss = 0.044644479943520346
Saved best model at epoch 1 with val loss 0.0446
epoch= 2/100, loss = 0.028900489595759173
Saved best model at epoch 2 with val loss 0.0289
epoch= 3/100, loss = 0.02940089043999172
epoch= 4/100, loss = 0.02506005302142433
Saved best model at epoch 4 with val loss 0.0251
epoch= 5/100, loss = 0.024889407345377233
Saved best model at epoch 5 with val loss 0.0249
epoch= 6/100, loss = 0.026022040113077863
epoch= 7/100, loss = 0.02319835789461175
Saved best model at epoch 7 with val loss 0.0232
epoch= 8/100, loss = 0.02558793467671975
epoch= 9/100, loss = 0.023319416426365144
epoch= 10/100, loss = 0.021609781117623916
Saved best model at epoch 10 with val loss 0.0216
epoch= 11/100, loss = 0.019492350599688034
Saved best model at epoch 11 with val loss 0.0195
epoch= 12/100, loss = 0.020428795731909897
epoch= 13/100, loss = 0.021262761202904032
epoch= 14/100, loss = 0.01766133082666151
Saved best model at epoch 14 with val loss 0.0177
epoch= 15/100, lo

In [29]:
# Evaluation

model.eval()

total = 0
correct = 0

with torch.no_grad():
  for xb, yb in test_loader:
    outputs = model(xb) #[1.1, 0.5, 2.1, 0.8...(7 values)] then max will be 2.1 can be calculated with toch.max
    _, predicted = torch.max(outputs, 1)

    correct += (predicted == yb).sum().item()
    total += yb.size(0)

  # print("Total : ", total)
  # print("correct : ", correct)
  print("accuracy : ", correct/total * 100)

accuracy :  98.46796657381616
